# Cleaning the orders dataframe

Read in data

In [1]:
import pandas as pd

In [6]:
path = '../data/data_raw/'
orders_orig = pd.read_csv(path + 'orders.csv')

In [3]:
orders = orders_orig.copy()

In [4]:
orders['state'].value_counts()

state
Shopping Basket    117809
Completed           46605
Place Order         40883
Pending             14379
Cancelled            7233
Name: count, dtype: int64

### Eyeball the data for issues to fix

In [48]:
orders.head()

,order_id,created_date,total_paid,state
0,241319,2017-01-02 13:35:40,44.99,Cancelled
1,241423,2017-11-06 13:10:02,136.15,Completed
2,242832,2017-12-31 17:40:03,15.76,Completed
3,243330,2017-02-16 10:59:38,84.98,Completed
4,243784,2017-11-24 13:35:19,157.86,Cancelled


In [49]:
print(orders.info())
orders.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB
None


order_id        0
created_date    0
total_paid      5
state           0
dtype: int64

- created_date is an object - should be datetime
- total_paid has 5 missing values

### Check for duplicates

Duplicated rows:

In [50]:
orders.duplicated().sum()

0

Duplicated order_ids:

In [51]:
orders['order_id'].duplicated().sum()

0

### Format created_date as datetime

In [52]:
orders['created_date_dt'] = pd.to_datetime(orders['created_date'])
print(orders.info())
orders.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   order_id         226909 non-null  int64         
 1   created_date     226909 non-null  object        
 2   total_paid       226904 non-null  float64       
 3   state            226909 non-null  object        
 4   created_date_dt  226909 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 8.7+ MB
None


,order_id,created_date,total_paid,state,created_date_dt
55018,354554,2017-05-10 16:42:03,35.98,Shopping Basket,2017-05-10 16:42:03
18744,318224,2017-01-31 14:11:00,0.00,Shopping Basket,2017-01-31 14:11:00
85712,385287,2017-08-02 08:10:02,60.79,Shopping Basket,2017-08-02 08:10:02
169202,469559,2017-12-22 12:07:40,359.00,Shopping Basket,2017-12-22 12:07:40
98097,397681,2017-09-10 20:12:13,845.31,Completed,2017-09-10 20:12:13
136122,435815,2017-11-23 20:43:04,0.00,Place Order,2017-11-23 20:43:04
156323,456547,2017-12-06 22:01:48,583.00,Shopping Basket,2017-12-06 22:01:48
164800,465098,2017-12-17 21:34:57,74.89,Shopping Basket,2017-12-17 21:34:57
204093,504581,2018-02-05 07:24:26,1073.99,Cancelled,2018-02-05 07:24:26
63754,363298,2017-06-07 19:23:48,409.00,Shopping Basket,2017-06-07 19:23:48


Looks good!

### Investigate missing values in total_paid

In [53]:
orders[orders['total_paid'].isna()]

,order_id,created_date,total_paid,state,created_date_dt
127701,427314,2017-11-20 18:54:39,NaN,Pending,2017-11-20 18:54:39
132013,431655,2017-11-22 12:15:24,NaN,Pending,2017-11-22 12:15:24
147316,447411,2017-11-27 10:32:37,NaN,Pending,2017-11-27 10:32:37
148833,448966,2017-11-27 18:54:15,NaN,Pending,2017-11-27 18:54:15
149434,449596,2017-11-27 21:52:08,NaN,Pending,2017-11-27 21:52:08


All NAs in total paid are pending orders from late November 2017

Do all pending orders have NA for total paid?

In [54]:
orders[orders['state']=='Pending']

,order_id,created_date,total_paid,state,created_date_dt
7,245851,2017-04-04 20:58:21,79.99,Pending,2017-04-04 20:58:21
17,252371,2017-02-09 12:31:57,27.98,Pending,2017-02-09 12:31:57
24,254537,2017-05-23 19:58:30,102.97,Pending,2017-05-23 19:58:30
27,256434,2018-02-26 07:32:21,39.99,Pending,2018-02-26 07:32:21
32,258087,2017-02-28 19:15:24,84.73,Pending,2017-02-28 19:15:24
...,...,...,...,...,...
226877,527370,2018-03-14 13:47:00,19.98,Pending,2018-03-14 13:47:00
226880,527373,2018-03-14 13:44:45,31.38,Pending,2018-03-14 13:44:45
226886,527379,2018-03-14 13:50:49,9.99,Pending,2018-03-14 13:50:49
226898,527391,2018-03-14 13:57:54,54.98,Pending,2018-03-14 13:57:54


No - only 5/14379 pending orders have NA for total paid

Replace NAs with the sum of the unit prices in orderlines?

In [55]:
# Read in orderlines
orderlines = pd.read_csv(path + 'orderlines.csv')
orderlines.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38


In [56]:
orderlines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


orderlines unit_price is string - reformat as numeric

In [57]:
#remove first '.' in unit_price
dot_mask = orderlines['unit_price'].str.count('\.')==2
orderlines.loc[dot_mask, 'unit_price'] = orderlines.loc[dot_mask, 'unit_price'].str.replace('.', '', 1)
orderlines['unit_price_num'] = pd.to_numeric(orderlines['unit_price'])
orderlines.sample(10)

,id,id_order,product_id,product_quantity,sku,unit_price,date,unit_price_num
171615,1453209,446061,0,1,IKM0049,85.49,2017-11-26 20:00:17,85.49
276198,1622119,515237,0,1,PIE0031,19.99,2018-02-21 09:03:32,19.99
279562,1627809,517736,0,1,WAC0152,16.60,2018-02-26 15:45:33,16.60
144701,1402653,424445,0,1,SEA0049,45.54,2017-11-10 11:45:51,45.54
158178,1430763,436035,0,1,OWC0057-2,35.69,2017-11-23 22:03:12,35.69
109118,1326361,393388,0,1,NTE0025,79.99,2017-08-28 15:05:25,79.99
112507,1344582,396278,0,1,JMO0085,26.99,2017-09-04 11:01:46,26.99
161883,1437266,439096,0,1,BEL0289,15.29,2017-11-24 13:37:01,15.29
227981,1544057,484462,0,1,APP1641,754.00,2018-01-07 19:09:38,754.00
253620,1585827,500587,0,1,PAC2075,2486.59,2018-01-29 03:00:04,2486.59


In [58]:
# multiply unit_price by quantity
orderlines['unit_price_total'] = orderlines['unit_price_num']*orderlines['product_quantity']
# aggregate orderlines and sum unit_prices
sum_price = pd.DataFrame(orderlines.groupby('id_order', as_index = False)['unit_price_total'].sum())
sum_price = sum_price.rename(columns = {'unit_price_total':'total_price_orderlines'})
sum_price



,id_order,total_price_orderlines
0,241319,44.99
1,241355,135.98
2,241423,129.16
3,242832,10.77
4,243330,77.99
...,...,...
204850,527397,42.99
204851,527398,42.99
204852,527399,141.58
204853,527400,19.98


In [59]:
print(len(orders))
orders['order_id'].nunique()


226909


226909

In [60]:
orders = orders.merge(sum_price, left_on='order_id', right_on = 'id_order', how = 'left')
print(len(orders))


226909


In [61]:
orders.groupby('state').sample(3)

,order_id,created_date,total_paid,state,created_date_dt,id_order,total_price_orderlines
166268,466578,2017-12-22 20:45:48,167.98,Cancelled,2017-12-22 20:45:48,466578.0,160.99
35850,335334,2017-03-15 12:21:26,293.98,Cancelled,2017-03-15 12:21:26,335334.0,288.99
52668,352163,2017-05-03 10:47:06,479.00,Cancelled,2017-05-03 10:47:06,352163.0,479.00
46636,346129,2017-04-17 14:10:20,299.97,Completed,2017-04-17 14:10:20,346129.0,292.98
20003,319483,2017-02-03 10:15:23,925.98,Completed,2017-02-03 10:15:23,319483.0,925.98
180673,481076,2018-01-03 16:24:41,109.00,Completed,2018-01-03 16:24:41,481076.0,109.00
179038,479431,2018-01-02 11:56:57,2133.59,Pending,2018-01-02 11:56:57,479431.0,2133.59
40018,339504,2017-03-27 03:55:36,415.99,Pending,2017-03-27 03:55:36,339504.0,415.99
215525,516016,2018-02-22 15:05:33,879.65,Pending,2018-02-22 15:05:33,516016.0,879.65
178710,479102,2018-01-02 04:38:51,54.00,Place Order,2018-01-02 04:38:51,479102.0,54.00


In [62]:
orders[orders['total_paid'].isna()]

,order_id,created_date,total_paid,state,created_date_dt,id_order,total_price_orderlines
127701,427314,2017-11-20 18:54:39,NaN,Pending,2017-11-20 18:54:39,427314.0,75.99
132013,431655,2017-11-22 12:15:24,NaN,Pending,2017-11-22 12:15:24,431655.0,154.00
147316,447411,2017-11-27 10:32:37,NaN,Pending,2017-11-27 10:32:37,447411.0,180.76
148833,448966,2017-11-27 18:54:15,NaN,Pending,2017-11-27 18:54:15,448966.0,961.02
149434,449596,2017-11-27 21:52:08,NaN,Pending,2017-11-27 21:52:08,449596.0,1722.59


In [63]:
orderlines[orderlines['id_order']==427314]

,id,id_order,product_id,product_quantity,sku,unit_price,date,unit_price_num,unit_price_total
150248,1416957,427314,0,1,MUJ0023,26.99,2017-11-20 18:40:45,26.99,26.99
150252,1416966,427314,0,1,APP1696,49.00,2017-11-20 18:44:30,49.00,49.00


In [64]:
(orders['total_paid'] - orders['total_price_orderlines']).abs().describe()

count    204691.000000
mean          2.928081
std         297.456334
min           0.000000
25%           0.000000
50%           0.000000
75%           0.010000
max       90898.610000
dtype: float64

In [65]:
maxdiff = (orders['total_paid'] - orders['total_price_orderlines']).abs().max()
orders[(orders['total_paid'] - orders['total_price_orderlines']).abs()==maxdiff]

,order_id,created_date,total_paid,state,created_date_dt,id_order,total_price_orderlines
77304,376851,2017-07-13 12:58:52,90989.6,Shopping Basket,2017-07-13 12:58:52,376851.0,90.99


total_price from orderlines seems to match (roughly) the total_paid in orders in most cases (75% of rows are within 3.99 EUR difference).
But some are very different (e.g. order_id 376851m still in shopping basket)


Maybe we can just use the total_price to fill in the NAs in total_paid in orders?

In [66]:
na_ids = [427314, 431655, 447411, 448966, 449596]
orders['total_paid'] = orders['total_paid'].fillna(orders['total_price_orderlines'])
orders[orders['order_id'].isin(na_ids)]


,order_id,created_date,total_paid,state,created_date_dt,id_order,total_price_orderlines
127701,427314,2017-11-20 18:54:39,75.99,Pending,2017-11-20 18:54:39,427314.0,75.99
132013,431655,2017-11-22 12:15:24,154.00,Pending,2017-11-22 12:15:24,431655.0,154.00
147316,447411,2017-11-27 10:32:37,180.76,Pending,2017-11-27 10:32:37,447411.0,180.76
148833,448966,2017-11-27 18:54:15,961.02,Pending,2017-11-27 18:54:15,448966.0,961.02
149434,449596,2017-11-27 21:52:08,1722.59,Pending,2017-11-27 21:52:08,449596.0,1722.59


### Save the cleaned csv


In [69]:
orders.to_csv('orders_cleaned.csv')